# Alpamayo 1.5: Navigation Prompt Sweep

This notebook tests a fixed set of navigation prompt templates on the same offline local inference window.

Goals:
1. Fix a single t0 window
2. Compare `no nav` against several left/right distance-based navigation templates
3. Plot all best trajectories on the same BEV figure
4. Print all CoTs
5. Sort the results to see which prompt changes the output the most


In [ ]:
import os
import sys
import textwrap
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import av
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)

print('repo_root =', repo_root)
print('src_path =', src_path)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)

# ===== Paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Camera =====
TARGET_CAMERA_FILENAME = 'Front_camera.mp4'
FRONT_FRAME_T = -1

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)
EVAL_XY_ROTATION_DEG = -90.0

# ===== Target t0 =====
# Set AUTO_PICK_LAST_VALID_T0=False if you want to manually override TARGET_T0_SOD.
AUTO_PICK_LAST_VALID_T0 = True
LAST_VALID_T0_OFFSET = 0
TARGET_T0_SOD = 175508.48

# ===== Prompt sweep =====
NAV_TEXTS = [
    'Turn right in 50m',
    'Turn right in 30m',
    'Turn right in 10m',
    'Turn right in 5m',
    'Turn left in 30m',
    'Turn left in 10m',
]

# ===== Display =====
PRINT_FULL_COT = True
COT_WRAP_WIDTH = 76
COT_MAX_CHARS_INLINE = 1200

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('SEED =', SEED)
print('NUM_TRAJ_SAMPLES =', NUM_TRAJ_SAMPLES)
print('AUTO_PICK_LAST_VALID_T0 =', AUTO_PICK_LAST_VALID_T0)
print('TARGET_T0_SOD =', TARGET_T0_SOD)
print('NAV_TEXTS =')
for s in NAV_TEXTS:
    print('  -', s)


### Helper functions


In [ ]:
def compute_max_valid_t0_range(cache, *, num_history_steps, num_future_steps, time_step, num_frames, fps, frame0_gps_time_sod):
    pose_min = float(cache.pose_times_sod[0])
    pose_max = float(cache.pose_times_sod[-1])

    history_left_span = (num_history_steps - 1) * time_step
    future_right_span = num_future_steps * time_step

    pose_t0_min = pose_min + history_left_span
    pose_t0_max = pose_max - future_right_span

    image_left_span = (num_frames - 1) * time_step
    image_t0_min = frame0_gps_time_sod + image_left_span
    return pose_t0_min, pose_t0_max, image_t0_min


def build_valid_t0_list(cache, *, num_history_steps, num_future_steps, time_step, num_frames, fps, frame0_gps_time_sod, step_sec=1.0):
    pose_t0_min, pose_t0_max, image_t0_min = compute_max_valid_t0_range(
        cache,
        num_history_steps=num_history_steps,
        num_future_steps=num_future_steps,
        time_step=time_step,
        num_frames=num_frames,
        fps=fps,
        frame0_gps_time_sod=frame0_gps_time_sod,
    )

    t0_min = max(pose_t0_min, image_t0_min)
    t0_max = pose_t0_max
    if t0_max < t0_min:
        raise ValueError(f'No valid t0 range. Computed range: [{t0_min:.3f}, {t0_max:.3f}]')

    return list(np.arange(t0_min, t0_max + 1e-9, step_sec, dtype=np.float64))


def pick_target_t0(cache) -> float:
    if not AUTO_PICK_LAST_VALID_T0:
        return float(TARGET_T0_SOD)

    t0_list = build_valid_t0_list(
        cache,
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        step_sec=1.0,
    )

    if len(t0_list) == 0:
        raise ValueError('No valid t0 found.')

    idx = max(0, len(t0_list) - 1 - int(LAST_VALID_T0_OFFSET))
    return float(t0_list[idx])


def truncate_text(s: str, max_chars: int) -> str:
    s = '' if s is None else str(s).strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + ' ...'


def wrap_text_block(s: str, width: int) -> str:
    s = '' if s is None else str(s).strip()
    return textwrap.fill(s, width=width)


def extract_metrics_row(result: dict) -> dict:
    df_metrics = result.get('df_metrics', None)
    if df_metrics is None or len(df_metrics) == 0:
        return {}
    return df_metrics.iloc[0].to_dict()


def decode_single_frame(video_path: Path, frame_index: int) -> np.ndarray:
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for idx, frame in enumerate(container.decode(stream)):
            if idx == frame_index:
                return frame.to_ndarray(format='rgb24')
    raise IndexError(f'Frame index {frame_index} out of range for {video_path}')


def pick_camera_row_by_exact_filename(clip_dir: Path, filename: str):
    camera_paths = helper.discover_offline_camera_files(clip_dir)
    camera_paths_sorted = sorted(camera_paths, key=lambda p: helper.infer_camera_index(p.name))
    matches = [(row, p) for row, p in enumerate(camera_paths_sorted) if p.name == filename]

    if len(matches) == 0:
        raise ValueError(f'No camera file matched exact filename={filename!r}')
    if len(matches) > 1:
        raise ValueError(f'Multiple camera files matched exact filename={filename!r}')

    row, path = matches[0]
    return row, path


def load_data_for_t0(t0_sod: float):
    return load_offline_dataset(
        clip_dir=CLIP_DIR,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=False,
        image_size=IMAGE_SIZE,
    )


def load_front_frame_for_data(data: dict, front_cam_row: int, front_video_path: Path):
    frame_idx = int(data['actual_video_frame_indices'][front_cam_row, FRONT_FRAME_T].item())
    frame_ts = float(data['absolute_timestamps_sod'][front_cam_row, FRONT_FRAME_T].item())
    img = decode_single_frame(front_video_path, frame_idx)
    return frame_idx, frame_ts, img


def run_single_condition_from_loaded_data(data: dict, *, nav_text, model, processor):
    result = run_offline_inference_window(
        data=data,
        model=model,
        processor=processor,
        device=DEVICE,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        time_step=TIME_STEP,
        eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
        nav_text=nav_text,
    )
    result['metrics_row'] = extract_metrics_row(result)
    return result


def run_prompt_sweep(t0_sod: float, *, nav_texts, model, processor, front_cam_row, front_video_path):
    data = load_data_for_t0(t0_sod)
    frame_idx, frame_ts, front_img = load_front_frame_for_data(data, front_cam_row, front_video_path)

    no_nav_result = run_single_condition_from_loaded_data(
        data,
        nav_text=None,
        model=model,
        processor=processor,
    )

    nav_results = {}
    for nav_text in nav_texts:
        nav_results[nav_text] = run_single_condition_from_loaded_data(
            data,
            nav_text=nav_text,
            model=model,
            processor=processor,
        )

    return {
        't0_sod': float(t0_sod),
        'data': data,
        'front_frame_idx': frame_idx,
        'front_frame_ts': frame_ts,
        'front_img': front_img,
        'no_nav': no_nav_result,
        'nav_results': nav_results,
    }


def plot_best_sample(ax, best_xyz, color, label, linewidth=2.4, markersize=3.5, alpha=0.95):
    xy = best_xyz[:, :2]
    ax.plot(
        xy[:, 0], xy[:, 1], 'o-',
        color=color,
        linewidth=linewidth,
        markersize=markersize,
        alpha=alpha,
        label=label,
    )


def render_prompt_sweep_figure(sweep_result):
    fig = plt.figure(figsize=(22, 8))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.4, 1.2])

    ax_img = fig.add_subplot(gs[0, 0])
    ax_bev = fig.add_subplot(gs[0, 1])
    ax_text = fig.add_subplot(gs[0, 2])

    no_nav = sweep_result['no_nav']
    gt_xyz = no_nav['gt_xyz_plot']
    hist_xyz = no_nav['hist_xyz_plot']

    all_arrays = [hist_xyz, gt_xyz, no_nav['pred_best_xyz']]
    for nav_text in NAV_TEXTS:
        all_arrays.append(sweep_result['nav_results'][nav_text]['pred_best_xyz'])

    xlim, ylim = _compute_adaptive_xy_limits(*all_arrays, min_span=20.0, pad_ratio=0.12, pad_min=2.0)

    ax_img.imshow(sweep_result['front_img'])
    ax_img.set_title(
        f"Front camera | frame={sweep_result['front_frame_idx']}\n"
        f"ts={sweep_result['front_frame_ts']:.3f}"
    )
    ax_img.axis('off')

    ax_bev.plot(hist_xyz[:, 0], hist_xyz[:, 1], 'b-o', linewidth=2.0, markersize=3.0, label='History')
    ax_bev.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=3.0, markersize=3.2, label='GT')
    plot_best_sample(ax_bev, no_nav['pred_best_xyz'], color='red', label='no nav')

    color_map = {
        'Turn right in 50m': 'tab:blue',
        'Turn right in 30m': 'tab:orange',
        'Turn right in 10m': 'tab:green',
        'Turn right in 5m': 'tab:purple',
        'Turn left in 30m': 'tab:brown',
        'Turn left in 10m': 'tab:pink',
    }

    for nav_text in NAV_TEXTS:
        nav_res = sweep_result['nav_results'][nav_text]
        plot_best_sample(ax_bev, nav_res['pred_best_xyz'], color=color_map.get(nav_text, 'gray'), label=nav_text, linewidth=2.0, markersize=3.0, alpha=0.95)

    ax_bev.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
    ax_bev.set_xlabel('x')
    ax_bev.set_ylabel('y')
    ax_bev.set_title(f"Prompt sweep best trajectories | t0={sweep_result['t0_sod']:.3f}")
    ax_bev.set_xlim(*xlim)
    ax_bev.set_ylim(*ylim)
    ax_bev.set_aspect('equal', adjustable='box')
    ax_bev.grid(True, alpha=0.3)
    ax_bev.legend(loc='best', fontsize=8)

    base_metrics = no_nav.get('metrics_row', {})
    text_blocks = []
    text_blocks.append(f"t0_sod: {sweep_result['t0_sod']:.3f}")
    text_blocks.append(f"no nav minADE: {base_metrics.get('minADE_m', 'N/A')}")
    text_blocks.append(f"no nav final: {base_metrics.get('final_point_error_m', 'N/A')}")
    text_blocks.append('')

    for nav_text in NAV_TEXTS:
        nav_metrics = sweep_result['nav_results'][nav_text].get('metrics_row', {})
        short_cot = truncate_text(sweep_result['nav_results'][nav_text].get('cot_value', ''), 120)
        text_blocks.append(nav_text)
        text_blocks.append(f"  minADE={nav_metrics.get('minADE_m', 'N/A')}, final={nav_metrics.get('final_point_error_m', 'N/A')}")
        text_blocks.append(f"  CoT={short_cot}")
        text_blocks.append('')

    ax_text.text(
        0.01, 0.99,
        '\n'.join(text_blocks),
        transform=ax_text.transAxes,
        va='top', ha='left', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.95),
    )
    ax_text.axis('off')
    ax_text.set_title('Metrics / short CoT summary')

    plt.tight_layout()
    return fig


def print_all_cots(sweep_result):
    print('\n===== CoT (no nav) =====')
    print(sweep_result['no_nav'].get('cot_value', ''))
    for nav_text in NAV_TEXTS:
        print(f'\n===== CoT ({nav_text}) =====')
        print(sweep_result['nav_results'][nav_text].get('cot_value', ''))


def final_point_delta_xy(base_best_xyz: np.ndarray, nav_best_xyz: np.ndarray):
    dx = float(nav_best_xyz[-1, 0] - base_best_xyz[-1, 0])
    dy = float(nav_best_xyz[-1, 1] - base_best_xyz[-1, 1])
    return dx, dy


def final_path_mean_distance(base_best_xyz: np.ndarray, nav_best_xyz: np.ndarray):
    n = min(len(base_best_xyz), len(nav_best_xyz))
    diff = nav_best_xyz[:n, :2] - base_best_xyz[:n, :2]
    dist = np.linalg.norm(diff, axis=1)
    return float(dist.mean())


def build_prompt_sweep_summary_table(sweep_result):
    rows = []
    base = sweep_result['no_nav']
    base_metrics = base.get('metrics_row', {})

    for nav_text in NAV_TEXTS:
        nav_res = sweep_result['nav_results'][nav_text]
        nav_metrics = nav_res.get('metrics_row', {})
        dx_final, dy_final = final_point_delta_xy(base['pred_best_xyz'], nav_res['pred_best_xyz'])
        mean_path_delta = final_path_mean_distance(base['pred_best_xyz'], nav_res['pred_best_xyz'])
        rows.append({
            't0_sod': sweep_result['t0_sod'],
            'front_frame_idx': sweep_result['front_frame_idx'],
            'front_frame_ts': sweep_result['front_frame_ts'],
            'nav_text': nav_text,
            'no_nav_minADE_m': base_metrics.get('minADE_m', np.nan),
            'with_nav_minADE_m': nav_metrics.get('minADE_m', np.nan),
            'no_nav_final_point_error_m': base_metrics.get('final_point_error_m', np.nan),
            'with_nav_final_point_error_m': nav_metrics.get('final_point_error_m', np.nan),
            'no_nav_pred_final_x': base_metrics.get('pred_final_x', np.nan),
            'no_nav_pred_final_y': base_metrics.get('pred_final_y', np.nan),
            'with_nav_pred_final_x': nav_metrics.get('pred_final_x', np.nan),
            'with_nav_pred_final_y': nav_metrics.get('pred_final_y', np.nan),
            'delta_final_x_nav_minus_no_nav': dx_final,
            'delta_final_y_nav_minus_no_nav': dy_final,
            'mean_path_delta_vs_no_nav': mean_path_delta,
            'no_nav_cot': str(base.get('cot_value', '')),
            'with_nav_cot': str(nav_res.get('cot_value', '')),
        })

    return pd.DataFrame(rows)


### Load cache, model, processor, and front camera path


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=False, force_rebuild=False)
front_cam_row, front_video_path = pick_camera_row_by_exact_filename(CLIP_DIR, TARGET_CAMERA_FILENAME)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Loaded cache, model, processor, and front camera path.')
print('front_cam_row =', front_cam_row)
print('front_video_path =', front_video_path)


### Pick the target t0


In [ ]:
valid_t0_list = build_valid_t0_list(
    clip_cache,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    step_sec=1.0,
)

print('num valid t0 =', len(valid_t0_list))
print('first few valid t0 =', valid_t0_list[:5])
print('last few valid t0 =', valid_t0_list[-5:])

target_t0 = pick_target_t0(clip_cache)
print('target_t0 =', target_t0)


### Run prompt sweep at the selected t0


In [ ]:
sweep_result = run_prompt_sweep(
    target_t0,
    nav_texts=NAV_TEXTS,
    model=model,
    processor=processor,
    front_cam_row=front_cam_row,
    front_video_path=front_video_path,
)

print('t0_sod =', sweep_result['t0_sod'])
print('front_frame_idx =', sweep_result['front_frame_idx'])
print('front_frame_ts =', sweep_result['front_frame_ts'])

base_metrics = sweep_result['no_nav'].get('metrics_row', {})
print(
    f"no_nav: best_sample_idx={base_metrics.get('best_sample_idx', 'N/A')}, "
    f"minADE_m={base_metrics.get('minADE_m', 'N/A')}, "
    f"final_point_error_m={base_metrics.get('final_point_error_m', 'N/A')}"
)

for nav_text in NAV_TEXTS:
    nav_metrics = sweep_result['nav_results'][nav_text].get('metrics_row', {})
    print(
        f"with_nav [{nav_text}]: "
        f"best_sample_idx={nav_metrics.get('best_sample_idx', 'N/A')}, "
        f"minADE_m={nav_metrics.get('minADE_m', 'N/A')}, "
        f"final_point_error_m={nav_metrics.get('final_point_error_m', 'N/A')}"
    )


### Compare all best trajectories on one plot


In [ ]:
fig = render_prompt_sweep_figure(sweep_result)
plt.show()
plt.close(fig)


### Print all CoTs


In [ ]:
if PRINT_FULL_COT:
    print_all_cots(sweep_result)
else:
    print('PRINT_FULL_COT = False, skipped full CoT printing.')


### Prompt sweep summary table


In [ ]:
df_prompt_sweep = build_prompt_sweep_summary_table(sweep_result)
display(df_prompt_sweep)


### Sorted views: which prompt changes the result the most?


In [ ]:
print('Sorted by mean_path_delta_vs_no_nav (descending):')
display(df_prompt_sweep.sort_values('mean_path_delta_vs_no_nav', ascending=False))

print('Sorted by absolute final-y change vs no-nav (descending):')
display(df_prompt_sweep.assign(abs_delta_final_y=lambda d: d['delta_final_y_nav_minus_no_nav'].abs()).sort_values('abs_delta_final_y', ascending=False))

print('Sorted by with_nav_minADE_m (ascending):')
display(df_prompt_sweep.sort_values('with_nav_minADE_m', ascending=True))
